In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import os

from pybedtools import BedTool

In [ ]:
projdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun'
pubdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc/csv/dmrs/ips_cluster/by_group'
pubfig = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
outdir = f'{projdir}/csv'
dmrdir = f'{projdir}/csv/dmrs/ips_cluster_cgn/by_group'
os.makedirs(dmrdir, exist_ok=True)

### early-iPSC methylpy DMRs (CGN pseudobulks), as in pseudobulk_allc/ipynb/dmrs/ips_cluster/01a_process_ips_cluster_dmrs

In [ ]:
dmrs = pd.read_csv(f'{projdir}/dmrs/methylpy/ips_cluster_cgn/_rms_results_collapsed.tsv', header=0, sep='\t')
dmrs['size'] = dmrs['end'] - dmrs['start']
dmrs.shape

In [ ]:
dmrs.head()

In [ ]:
dmrs['hypermethylated_samples'].value_counts(), dmrs['hypomethylated_samples'].value_counts()

In [ ]:
dmrs['hypermethylated_samples'] = dmrs['hypermethylated_samples'].str.split(',')
dmrs['hypomethylated_samples'] = dmrs['hypomethylated_samples'].str.split(',')
dmrs = dmrs.explode('hypermethylated_samples').reset_index(drop=True)
dmrs = dmrs.explode('hypomethylated_samples').reset_index(drop=True)
dmrs.index = [f'{x}_{y}' for x,y in zip(dmrs['#chr'], dmrs['start'])]
dmrs.shape

In [ ]:
counts = {}
for s in ['IPS_main', 'IPS_alt']:
    hypo_dmrs = dmrs[dmrs['hypomethylated_samples'] == s]
    hyper_dmrs = dmrs[dmrs['hypermethylated_samples'] == s]
    hypo_dmrs = hypo_dmrs[~hypo_dmrs.index.duplicated()]
    hyper_dmrs = hyper_dmrs[~hyper_dmrs.index.duplicated()]
    hypo_dmrs.to_csv(f'{dmrdir}/{s}_hypo.csv', sep='\t', header=None, index=None)
    hyper_dmrs.to_csv(f'{dmrdir}/{s}_hyper.csv', sep='\t', header=None, index=None)
    final_hypo = hypo_dmrs[hypo_dmrs['number_of_dms'] > 1]
    final_hyper = hyper_dmrs[hyper_dmrs['number_of_dms'] > 1]
    final_hypo.to_csv(f'{dmrdir}/{s}_hypo_dms1.csv', sep='\t', header=None, index=None)
    final_hyper.to_csv(f'{dmrdir}/{s}_hyper_dms1.csv', sep='\t', header=None, index=None)
    counts[s] = [len(hypo_dmrs), len(hyper_dmrs), len(final_hypo), len(final_hyper)]
counts = pd.DataFrame(counts, index=['hypo', 'hyper', 'hypo_dms1', 'hyper_dms1']).T
counts

### compare with published (all-nuclei) DMR counts

In [ ]:
pub_counts = {}
for s in ['IPS_main', 'IPS_alt']:
    pub_counts[s] = [sum(1 for _ in open(f'{pubdir}/{s}_{t}.csv')) for t in ['hypo', 'hyper', 'hypo_dms1', 'hyper_dms1']]
pub_counts = pd.DataFrame(pub_counts, index=['hypo', 'hyper', 'hypo_dms1', 'hyper_dms1']).T
pub_counts

### overlap with early-iPSC differential TAD boundaries, as in 01d_overlap_with_tad_boundaries

In [ ]:
hypo = pd.read_csv(f'{dmrdir}/IPS_alt_hypo_dms1.csv', sep='\t', header=None)
hyper = pd.read_csv(f'{dmrdir}/IPS_alt_hyper_dms1.csv', sep='\t', header=None)
hypo.shape, hyper.shape

In [ ]:
hypo.head()

In [ ]:
diff = pd.read_csv(f'{outdir}/ips_cluster_diff_bins.csv', sep='\t', index_col=0)
nondiff = pd.read_csv(f'{outdir}/ips_cluster_nondiff_bins.csv', sep='\t', index_col=0)
diff.shape, nondiff.shape

In [ ]:
hypo_bt = BedTool.from_dataframe(hypo)
hyper_bt = BedTool.from_dataframe(hyper)
diff_bt = BedTool.from_dataframe(diff)
nondiff_bt = BedTool.from_dataframe(nondiff)

n_hypo_diff = len(hypo_bt.intersect(diff_bt, u=True))
n_hyper_diff = len(hyper_bt.intersect(diff_bt, u=True))
n_hypo_nondiff = len(hypo_bt.intersect(nondiff_bt, u=True))
n_hyper_nondiff = len(hyper_bt.intersect(nondiff_bt, u=True))
n_hypo_diff, n_hyper_diff, n_hypo_nondiff, n_hyper_nondiff

In [ ]:
to_plot = pd.DataFrame()
to_plot['overlap'] = [n_hypo_diff / hypo.shape[0], n_hyper_diff / hyper.shape[0],
                      n_hypo_nondiff / hypo.shape[0], n_hyper_nondiff / hyper.shape[0]]
to_plot['dmr_type'] = np.tile(['hypo', 'hyper'], 2)
to_plot['bin_type'] = np.repeat(['diff', 'non-diff'], 2)
to_plot['n'] = [n_hypo_diff, n_hyper_diff, n_hypo_nondiff, n_hyper_nondiff]
to_plot['n_dmrs'] = [hypo.shape[0], hyper.shape[0]] * 2
to_plot

### published

In [ ]:
pd.read_csv(f'{pubfig}/ips_subtype_dmr_tad_boundary_overlap.csv', sep='\t', index_col=0)

### background: fraction of tested 25kb bins that are differential

In [ ]:
diff.shape[0] / (diff.shape[0] + nondiff.shape[0])

In [ ]:
to_plot.to_csv(f'{outdir}/ips_subtype_dmr_tad_boundary_overlap.csv', sep='\t')

In [ ]:
!date